# Name: Baraka Mandi

# Question 3: Convolution Operations with Different Parameters

**Task:** Perform convolution on a 5×5 input matrix with a 3×3 kernel using different stride and padding settings, implemented with both **NumPy** (from scratch) and **PyTorch**.

**Padding definitions:**
- **VALID** – no padding; output size = ⌊(N − K) / S⌋ + 1
- **SAME** – zero-pad so that output size = ⌈N / S⌉; total padding = max((out − 1)·S + K − N, 0), split between both sides (extra goes to bottom/right)

In [1]:
import numpy as np
import torch
import torch.nn.functional as F

## 1. Define the 5×5 input matrix

In [2]:
input_matrix = np.array([
    [ 1,  2,  3,  4,  5],
    [ 6,  7,  8,  9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20],
    [21, 22, 23, 24, 25]
], dtype=np.float32)

print("Input Matrix (5x5):")
print(input_matrix)

Input Matrix (5x5):
[[ 1.  2.  3.  4.  5.]
 [ 6.  7.  8.  9. 10.]
 [11. 12. 13. 14. 15.]
 [16. 17. 18. 19. 20.]
 [21. 22. 23. 24. 25.]]


## 2. Define the 3×3 kernel

In [3]:
kernel = np.array([
    [0,  1, 0],
    [1, -4, 1],
    [0,  1, 0]
], dtype=np.float32)

print("Kernel (3x3):")
print(kernel)

Kernel (3x3):
[[ 0.  1.  0.]
 [ 1. -4.  1.]
 [ 0.  1.  0.]]


## 3a. Helper: compute padding for VALID / SAME

In [4]:
def compute_padding(in_size, k_size, stride, padding):
    """Return (pad_before, pad_after, out_size) for one spatial dimension."""
    if padding == 'VALID':
        out_size = (in_size - k_size) // stride + 1
        return 0, 0, out_size
    elif padding == 'SAME':
        out_size = int(np.ceil(in_size / stride))
        pad_total = max((out_size - 1) * stride + k_size - in_size, 0)
        pad_before = pad_total // 2
        pad_after = pad_total - pad_before
        return pad_before, pad_after, out_size
    else:
        raise ValueError("padding must be 'VALID' or 'SAME'")

## 3b. Convolution with NumPy (from scratch)

In [5]:
def conv2d_numpy(x, k, stride=1, padding='VALID'):
    kh, kw = k.shape
    H, W = x.shape

    pt, pb, out_h = compute_padding(H, kh, stride, padding)
    pl, pr, out_w = compute_padding(W, kw, stride, padding)

    x_padded = np.pad(x, ((pt, pb), (pl, pr)), mode='constant', constant_values=0)

    output = np.zeros((out_h, out_w), dtype=np.float32)
    for i in range(out_h):
        for j in range(out_w):
            r, c = i * stride, j * stride
            region = x_padded[r:r + kh, c:c + kw]
            output[i, j] = np.sum(region * k)
    return output

## 3c. Convolution with PyTorch

In [6]:
def conv2d_torch(x, k, stride=1, padding='VALID'):
    kh, kw = k.shape
    H, W = x.shape

    # PyTorch expects (batch, channels, height, width)
    x_t = torch.tensor(x).reshape(1, 1, H, W)
    k_t = torch.tensor(k).reshape(1, 1, kh, kw)

    pt, pb, _ = compute_padding(H, kh, stride, padding)
    pl, pr, _ = compute_padding(W, kw, stride, padding)

    # F.pad order is (left, right, top, bottom). Explicit padding is used because
    # PyTorch's built-in padding='same' does not support stride > 1.
    x_t = F.pad(x_t, (pl, pr, pt, pb), mode='constant', value=0)

    out = F.conv2d(x_t, k_t, stride=stride, padding=0)
    return out.squeeze().numpy()

## 4. Perform convolutions and print output feature maps

In [7]:
configs = [
    (1, 'VALID'),
    (1, 'SAME'),
    (2, 'VALID'),
    (2, 'SAME'),
]

for stride, padding in configs:
    out_np = conv2d_numpy(input_matrix, kernel, stride, padding)
    out_pt = conv2d_torch(input_matrix, kernel, stride, padding)

    print("=" * 45)
    print(f"Stride = {stride}, Padding = '{padding}'   ->  Output shape: {out_pt.shape}")
    print("=" * 45)
    print("NumPy output feature map:")
    print(out_np)
    print("\nPyTorch output feature map:")
    print(out_pt)
    print("\nNumPy and PyTorch match:", np.allclose(out_np, out_pt))
    print()

Stride = 1, Padding = 'VALID'   ->  Output shape: (3, 3)
NumPy output feature map:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

PyTorch output feature map:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

NumPy and PyTorch match: True

Stride = 1, Padding = 'SAME'   ->  Output shape: (5, 5)
NumPy output feature map:
[[  4.   3.   2.   1.  -6.]
 [ -5.   0.   0.   0. -11.]
 [-10.   0.   0.   0. -16.]
 [-15.   0.   0.   0. -21.]
 [-46. -27. -28. -29. -56.]]

PyTorch output feature map:
[[  4.   3.   2.   1.  -6.]
 [ -5.   0.   0.   0. -11.]
 [-10.   0.   0.   0. -16.]
 [-15.   0.   0.   0. -21.]
 [-46. -27. -28. -29. -56.]]

NumPy and PyTorch match: True

Stride = 2, Padding = 'VALID'   ->  Output shape: (2, 2)
NumPy output feature map:
[[0. 0.]
 [0. 0.]]

PyTorch output feature map:
[[0. 0.]
 [0. 0.]]

NumPy and PyTorch match: True

Stride = 2, Padding = 'SAME'   ->  Output shape: (3, 3)
NumPy output feature map:
[[  4.   2.  -6.]
 [-10.   0. -16.]
 [-46. -28. -56.]]

PyTorch output feature map:
[[  4.   